# ROBERTA FINE-TUNING

In [ ]:
from collections import Counter
import json
from pathlib import Path
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Biometrics selected from exploration study
BIOMETRIC_FEATURES = [
    'blinks_duration_std_ns',
    'saccades_duration_mean_ns',
    '3d_eye_states_pupil diameter left [mm]_std',
    'fixations_duration_mean_ns',
    'fixations_duration_std_ns',
    'fixations_duration_max_ns',
    'saccades_duration_min_ns',
    'blinks_count',
    'reaction_time',
    'garmin_hr_min'
    # EEG optional (commented by default)
    # 'EXG_Channel_8_Delta_power'
]

def _feature_modality(feature_name):
    if feature_name.startswith('garmin_'):
        return 'HR'
    if feature_name.startswith('EXG_'):
        return 'EEG'
    return 'ET'

def extract_biometrics(sample, features=BIOMETRIC_FEATURES):
    mods = sample.get('sensorial', {}).get('modalities', {})
    values = []
    for f in features:
        modality = _feature_modality(f)
        by_user = mods.get(modality, {}).get('by_user', {})
        vals = []
        for _, feats in by_user.items():
            if f in feats:
                vals.append(feats[f])
        values.append(float(np.mean(vals)) if vals else np.nan)
    return values

def cargar_datos_entrenamiento(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    X = []
    y = []
    ids = []
    bios = []

    for _, value in data.items():
        tweet_text = value.get('text', '')
        votos = value.get('labels_task2_1', [])
        if not votos:
            continue
        conteo = Counter(votos)
        label_mayoritaria, frecuencia = conteo.most_common(1)[0]
        total_votos = len(votos)
        if frecuencia <= total_votos / 2:
            continue

        X.append(tweet_text)
        y.append(label_mayoritaria)
        ids.append(value['id_EXIST'])
        bios.append(extract_biometrics(value))

    return X, y, ids, bios

ruta = 'lab2_materials/dataset_task2_exist2026/training.json'
X_train, y_train, ids_train, bio_train = cargar_datos_entrenamiento(ruta)

# Normalize biometrics with train statistics only
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
bio_train_imp = imputer.fit_transform(bio_train)
bio_train_scaled = scaler.fit_transform(bio_train_imp)

print('Train samples:', len(X_train))
print('Biometric features:', len(BIOMETRIC_FEATURES))


Procesados 6064 tweets correctamente.
Ejemplo - Texto: @TheChiflis Ignora al otro, es un capullo.El probl...
Ejemplo - Label: YES


In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
import evaluate

model_id = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_id)

label_map = {'YES': 1, 'NO': 0}
train_dict = {
    'text': X_train,
    'label': [label_map[l] for l in y_train],
    'biometrics': bio_train_scaled.tolist()
}
ds = Dataset.from_dict(train_dict)

def preprocess_function(examples):
    result = tokenizer(examples['text'], truncation=True, padding=True, max_length=128)
    result['biometrics'] = examples['biometrics']
    return result

tokenized_ds = ds.map(preprocess_function, batched=True, remove_columns=['text'])

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['query', 'value']
)

class RobertaWithBiometrics(torch.nn.Module):
    def __init__(self, model_id, num_bio_features, peft_cfg=None):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_id)
        base = AutoModel.from_pretrained(model_id)
        if peft_cfg is not None:
            base = get_peft_model(base, peft_cfg)
        self.roberta = base
        hidden = self.roberta.config.hidden_size
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(hidden + num_bio_features, 2)

    def forward(self, input_ids=None, attention_mask=None, biometrics=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        cls = outputs.last_hidden_state[:, 0]
        if biometrics is None:
            raise ValueError('biometrics tensor is required')
        x = torch.cat([cls, biometrics], dim=1)
        x = self.dropout(x)
        logits = self.classifier(x)
        loss = None
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)

class DataCollatorWithBiometrics:
    def __init__(self, tokenizer):
        self.text_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def __call__(self, features):
        biometrics = torch.tensor([f['biometrics'] for f in features], dtype=torch.float)
        text_features = [{k: v for k, v in f.items() if k != 'biometrics'} for f in features]
        batch = self.text_collator(text_features)
        batch['biometrics'] = biometrics
        return batch

data_collator = DataCollatorWithBiometrics(tokenizer)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    metric = evaluate.load('f1')
    return metric.compute(predictions=predictions, references=labels, average='macro')

def model_init(trial=None):
    return RobertaWithBiometrics(model_id, num_bio_features=len(BIOMETRIC_FEATURES), peft_cfg=peft_config)


Map: 100%|██████████| 6064/6064 [00:00<00:00, 12673.91 examples/s]


In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold
import numpy as np
import evaluate

metric = evaluate.load('f1')
# Split 80/20 before training
split_ds = tokenized_ds.train_test_split(test_size=0.2, seed=42)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels, average='macro')

def model_init(trial=None):
    return RobertaWithBiometrics(model_id, num_bio_features=len(BIOMETRIC_FEATURES), peft_cfg=peft_config)

def optuna_hp_space(trial):
    return {
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 5e-5, log=True),
        'num_train_epochs': trial.suggest_int('num_train_epochs', 2, 5),
        'per_device_train_batch_size': trial.suggest_categorical('per_device_train_batch_size', [8, 16, 32])
    }

training_args = TrainingArguments(
    output_dir='./modelos_optuna',
    evaluation_strategy='epoch',
    save_strategy='no',
    fp16=True if torch.cuda.is_available() else False,
    logging_strategy='epoch'
)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=split_ds['train'],
    eval_dataset=split_ds['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

best_run = trainer.hyperparameter_search(
    direction='maximize',
    hp_space=optuna_hp_space,
    n_trials=10
)
print('Best run:', best_run)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1257.28it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- M

Iniciando búsqueda de hiperparámetros...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1238.23it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Con

Epoch,Training Loss,Validation Loss,F1
1,0.487067,0.426265,0.803610
2,0.452518,0.402363,0.817902
3,0.372585,0.400752,0.823697


[I 2026-03-04 10:48:50,442] Trial 0 finished with value: 0.8236966954614013 and parameters: {'learning_rate': 0.000448218550566169, 'num_train_epochs': 3, 'per_device_train_batch_size': 32}. Best is trial 0 with value: 0.8236966954614013.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1299.70it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.ou

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Fold 4 F1: 0.8385

---VALIDANDO FOLD 5 ---


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1261.22it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Con

Epoch,Training Loss,Validation Loss,F1
1,0.477417,0.476271,0.773280
2,0.416234,0.553552,0.742018
3,0.419454,0.410426,0.822030


Fold 5 F1: 0.8220

RESULTADO FINAL CV: 0.8198 (+/- 0.0114)


In [ ]:
import json
import torch
import numpy as np
from datasets import Dataset

print('Reentrenando modelo final con el dataset completo...')

final_training_args = TrainingArguments(
    output_dir='./modelo_final_exist',
    learning_rate=best_run.hyperparameters['learning_rate'],
    num_train_epochs=best_run.hyperparameters['num_train_epochs'],
    per_device_train_batch_size=best_run.hyperparameters['per_device_train_batch_size'],
    warmup_ratio=0.1,
    fp16=True if torch.cuda.is_available() else False,
    save_strategy='no'
)

trainer_final = Trainer(
    model_init=model_init,
    args=final_training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)

trainer_final.train()

# Load test data
ruta_test = 'lab2_materials/dataset_task2_exist2026/test.json'
with open(ruta_test, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

ids_test = []
textos_test = []
bio_test = []

for _, value in test_data.items():
    ids_test.append(value['id_EXIST'])
    textos_test.append(value.get('text', ''))
    bio_test.append(extract_biometrics(value))

# Normalize biometrics using train statistics
bio_test_imp = imputer.transform(bio_test)
bio_test_scaled = scaler.transform(bio_test_imp)

test_dict = {
    'text': textos_test,
    'biometrics': bio_test_scaled.tolist()
}
test_ds = Dataset.from_dict(test_dict)
tokenized_test = test_ds.map(preprocess_function, batched=True, remove_columns=['text'])

print('Generando predicciones para el test oficial...')
preds = trainer_final.predict(tokenized_test).predictions
preds = np.argmax(preds, axis=1)

label_map_inv = {1: 'YES', 0: 'NO'}
output_json = []
for id_ev, pred_idx in zip(ids_test, preds):
    output_json.append({
        'test_case': 'EXIST2026',
        'id': id_ev,
        'value': label_map_inv[pred_idx]
    })

nombre_archivo = 'Thinkpol_modelo_ROBERTA_LoRA_biometrics.json'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    json.dump(output_json, f, ensure_ascii=False, indent=4)

print(f'Proceso finalizado. Archivo {nombre_archivo} listo para entrega.')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Reentrenando modelo final con el dataset completo...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1506.60it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Con

Step,Training Loss
500,0.489377


Generando predicciones para el test oficial...
Proceso finalizado. Archivo Thinkpol_modelo_ROBERTA_LoRA.json listo para entrega.
